In [0]:
import requests
import json
import time
from datetime import datetime

# Criar schema prata e registrar views da bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS prata")

spark.sql("SELECT * FROM bronze.eventos").createOrReplaceTempView("bronze_eventos_view")
spark.sql("SELECT * FROM bronze.eventos_presenca").createOrReplaceTempView("bronze_presenca_view")

print("Schema e views criados.")

In [0]:
%sql
CREATE OR REPLACE TABLE prata.eventos
USING DELTA
COMMENT 'Camada Prata - Eventos legislativos limpos e dimensionados'
AS
SELECT 
    id_evento,
    CAST(data_hora_inicio AS TIMESTAMP) AS data_hora_inicio,
    CAST(data_hora_fim AS TIMESTAMP) AS data_hora_fim,
    situacao,
    descricao_tipo,
    descricao,
    local_externo,
    url_registro,
    get_json_object(orgaos_json, '$[0].sigla') AS sigla_orgao,
    get_json_object(orgaos_json, '$[0].nome') AS nome_orgao,
    get_json_object(orgaos_json, '$[0].codTipoOrgao') AS cod_tipo_orgao,
    get_json_object(local_camara_json, '$.nome') AS local_camara_nome,
    YEAR(CAST(data_hora_inicio AS DATE)) AS ano,
    MONTH(CAST(data_hora_inicio AS DATE)) AS mes,
    DAYOFWEEK(CAST(data_hora_inicio AS DATE)) AS dia_semana,
    WEEKOFYEAR(CAST(data_hora_inicio AS DATE)) AS semana_ano,
    YEAR(CAST(data_hora_inicio AS DATE)) * 100 + WEEKOFYEAR(CAST(data_hora_inicio AS DATE)) AS chave_semana,
    data_coleta,
    origem
FROM bronze_eventos_view

# Verificação: prata.eventos
Conferir totais, tipos de evento, órgãos e período.

In [0]:
%sql
SELECT 
    COUNT(*) AS total,
    COUNT(DISTINCT descricao_tipo) AS tipos_evento,
    COUNT(DISTINCT sigla_orgao) AS orgaos,
    MIN(ano) AS ano_min,
    MAX(ano) AS ano_max,
    COUNT(DISTINCT semana_ano) AS semanas
FROM prata.eventos


# Tabela prata.eventos_presenca
Presenças limpas com dimensões de data e join com eventos.

In [0]:
%sql
CREATE OR REPLACE TABLE prata.eventos_presenca
USING DELTA
COMMENT 'Camada Prata - Presenca de deputados em eventos limpos e dimensionados'
AS
SELECT 
    p.id_deputado,
    p.nome_deputado,
    p.sigla_partido,
    p.sigla_uf,
    p.id_evento,
    CAST(p.data_hora_inicio AS TIMESTAMP) AS data_hora_inicio,
    CAST(p.data_hora_fim AS TIMESTAMP) AS data_hora_fim,
    p.descricao_tipo,
    p.situacao,
    YEAR(CAST(p.data_hora_inicio AS DATE)) AS ano,
    MONTH(CAST(p.data_hora_inicio AS DATE)) AS mes,
    DAYOFWEEK(CAST(p.data_hora_inicio AS DATE)) AS dia_semana,
    WEEKOFYEAR(CAST(p.data_hora_inicio AS DATE)) AS semana_ano,
    YEAR(CAST(p.data_hora_inicio AS DATE)) * 100 + WEEKOFYEAR(CAST(p.data_hora_inicio AS DATE)) AS chave_semana,
    e.sigla_orgao,
    e.nome_orgao,
    p.data_coleta,
    p.origem
FROM bronze_presenca_view p
LEFT JOIN prata.eventos e ON p.id_evento = e.id_evento

# Verificação: prata.eventos_presenca
Conferir totais, deputados, eventos, tipos e órgãos.

In [0]:
%sql
SELECT 
    COUNT(*) AS total,
    COUNT(DISTINCT id_deputado) AS deputados,
    COUNT(DISTINCT id_evento) AS eventos,
    COUNT(DISTINCT descricao_tipo) AS tipos_evento,
    COUNT(DISTINCT sigla_orgao) AS orgaos,
    COUNT(DISTINCT ano) AS anos
FROM prata.eventos_presenca